# Assignment 7: Classification Analysis | Pair Assignment

**Goal:** build and compare a **Decision Tree** and a **Naive Bayes** classifier on your
group's dataset, then report what you found. EAch pair will choose one dataset from the following table and do the following.


Hints are provided in the cells marked `# TODO`. If you get stuck on syntax, check the code examples form this week.

> **Replace the loan_sample.cs with your chosen Kaggle dataset** and update the target column.

## Your dataset

| Group dataset | Example classification task | Target column | Kaggle link |
|---|---|---|---|
| SF Crime | Predict crime category, or district | `Category` / `PdDistrict` | https://www.kaggle.com/competitions/sf-crime/data
| NASA Asteroids | Predict if an asteroid is hazardous | `Hazardous` | https://www.kaggle.com/datasets/shrutimehta/nasa-asteroids-classification
| Hotel Booking | Predict if a booking is cancelled, or hotel type | `is_canceled` / `hotel` | https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Use a Decision Tree for one task and Naive Bayes for the other (or, for Asteroids, run
both on the same task and compare). Keep notes as you go; you will need them for the report
at the bottom.

## Know your dataset before you start | Each pair will choose one of the following dataset

Read the block for your group's dataset first. It tells you what you are actually
predicting and the specific things to watch for at each step, so you are not surprised
halfway through.

### SF Crime, predicting crime category or district

**What you are figuring out:** can the time and place of an incident predict what kind of
crime it was, or which police district it happened in? You are turning raw incident logs
into useful predictors.

- **Exploration:** this file is huge (about 1 million rows). Take a random sample (say 50,000 rows) so your code runs fast. Check how many distinct crime categories exist.
- **Preprocessing:** the timestamp is not useful as text. Pull `hour`, `day of week`, and `month` out of it into their own numeric columns. If there are too many crime categories, group the rare ones into an "Other" bucket so the model is not splitting on dozens of tiny classes. Keep latitude and longitude; they are strong predictors.
- **Build models:** Decision Tree for one task (for example, category) and Naive Bayes for the other (district).
- **Evaluate:** with many classes, accuracy alone is thin. Look at the confusion matrix to see which categories get mixed up.

In [85]:
# Prerequisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Ready.")

Ready.


## Step 1 - Data exploration

Load your data and get to know it: how many rows and columns, what the columns mean, and
how balanced the classes are. Class balance matters: if 90% of rows are one class, a model
can look accurate just by guessing that class every time.

In [86]:
# GIVEN: load the dataset. Change this path to your group's file when you have it.
df = pd.read_csv("train.csv")

# Set this to the name of the column you are predicting.
# time and place of an incident predict what kind of crime it was? 

target = "Category"

df.head()

,Dates,Category,Descript,DayOfWeek,PdDistrict,Resolution,Address,X,Y
0,2015-05-13 23:53:00,WARRANTS,WARRANT ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",OAK ST / LAGUNA ST,-122.425892,37.774599
1,2015-05-13 23:53:00,OTHER OFFENSES,TRAFFIC VIOLATION ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",OAK ST / LAGUNA ST,-122.425892,37.774599
2,2015-05-13 23:33:00,OTHER OFFENSES,TRAFFIC VIOLATION ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",VANNESS AV / GREENWICH ST,-122.424363,37.800414
3,2015-05-13 23:30:00,LARCENY/THEFT,GRAND THEFT FROM LOCKED AUTO,Wednesday,NORTHERN,NONE,1500 Block of LOMBARD ST,-122.426995,37.800873
4,2015-05-13 23:30:00,LARCENY/THEFT,GRAND THEFT FROM LOCKED AUTO,Wednesday,PARK,NONE,100 Block of BRODERICK ST,-122.438738,37.771541


In [ ]:
# TODO: explore the data
# Hint: df.shape, df.info(), and df[target].value_counts()
# Also try df.describe() for the numeric columns.

df.shape
df.info()
df[target].value_counts()
df.describe()

print (df.isna().sum())



## Step 2 - Preprocessing

Get the data ready for the models. You will typically need to:

- Drop columns that are just identifiers or are not useful (e.g. an ID column).
- Handle missing values (fill numeric columns with the median, text columns with the mode).
- Encode categorical (text) columns into numbers with `pd.get_dummies(...)` or `LabelEncoder`.
- Separate the target `y` from the features `X`, then split into train and test sets.

In [87]:
[df[col].value_counts() for col in df.columns]


[Dates
 2011-01-01 00:01:00    185
 2006-01-01 00:01:00    136
 2012-01-01 00:01:00     94
 2006-01-01 12:00:00     63
 2007-06-01 00:01:00     61
                       ... 
 2003-01-06 00:42:00      1
 2003-01-06 00:40:00      1
 2003-01-06 00:33:00      1
 2003-01-06 00:31:00      1
 2003-01-06 00:15:00      1
 Name: count, Length: 389257, dtype: int64,
 Category
 LARCENY/THEFT                  174900
 OTHER OFFENSES                 126182
 NON-CRIMINAL                    92304
 ASSAULT                         76876
 DRUG/NARCOTIC                   53971
 VEHICLE THEFT                   53781
 VANDALISM                       44725
 WARRANTS                        42214
 BURGLARY                        36755
 SUSPICIOUS OCC                  31414
 MISSING PERSON                  25989
 ROBBERY                         23000
 FRAUD                           16679
 FORGERY/COUNTERFEITING          10609
 SECONDARY CODES                  9985
 WEAPON LAWS                      8555
 PROSTI

In [ ]:
#drops useless columns- Resolurion, coordinates 

data = df.drop(columns=["Resolution","Address", "Descript"]).copy()

data["Hour"] = pd.to_datetime(data["Dates"]).dt.hour
data["Month"] = pd.to_datetime(data["Dates"]).dt.month

data.head()




,Dates,Category,DayOfWeek,PdDistrict,X,Y,Hour,Month
0,2015-05-13 23:53:00,WARRANTS,Wednesday,NORTHERN,-122.425892,37.774599,23,5
1,2015-05-13 23:53:00,OTHER OFFENSES,Wednesday,NORTHERN,-122.425892,37.774599,23,5
2,2015-05-13 23:33:00,OTHER OFFENSES,Wednesday,NORTHERN,-122.424363,37.800414,23,5
3,2015-05-13 23:30:00,LARCENY/THEFT,Wednesday,NORTHERN,-122.426995,37.800873,23,5
4,2015-05-13 23:30:00,LARCENY/THEFT,Wednesday,PARK,-122.438738,37.771541,23,5


In [ ]:
#encoding 

maps = {

    "Category": 
        {"LARCENY/THEFT": 0, "OTHER OFFENSES": 1, "NON-CRIMINAL": 2, "ASSAULT": 3, "DRUG/NARCOTIC": 4, "VEHICLE THEFT": 5, "VANDALISM": 6, "WARRANTS": 7, "BURGLARY": 8, "SUSPICIOUS OCC": 9, "MISSING PERSON": 10, "ROBBERY": 11, "FRAUD": 12, "FORGERY/COUNTERFEITING": 13, "SECONDARY CODES": 14, "WEAPON LAWS": 15, "PROSTITUTION": 16, "TRESPASS": 17, "STOLEN PROPERTY": 18, "SEX OFFENSES FORCIBLE": 19, "DISORDERLY CONDUCT": 20, "DRUNKENNESS": 21, "RECOVERED VEHICLE": 22, "KIDNAPPING": 23, "DRIVING UNDER THE INFLUENCE": 24, "RUNAWAY": 25, "LIQUOR LAWS": 26, "ARSON": 27, "LOITERING": 28, "EMBEZZLEMENT": 29, "SUICIDE": 30, "FAMILY OFFENSES": 31, "BAD CHECKS": 32, "BRIBERY": 33, "EXTORTION": 34, "SEX OFFENSES NON FORCIBLE": 35, "GAMBLING": 36, "PORNOGRAPHY/OBSCENE MAT": 37, "TREA": 38}, 

    "PdDistrict":
        {"SOUTHERN": 0,
         "MISSION": 1,
         "NORTHERN": 2,
         "BAYVIEW": 3,
         "CENTRAL": 4,
         "TENDERLOIN": 5,
         "INGLESIDE": 6,
         "TARAVAL": 7,
         "PARK": 8,
         "RICHMOND": 9},

    "DayOfWeek":
        {"Monday": 0,
         "Tuesday": 1,
         "Wednesday": 2,
         "Thursday": 3,
         "Friday": 4,
         "Saturday": 5,
         "Sunday": 6}, 

    "Month":
    {1:1, 2:2,3:3,4:4,5:5,6:6,7:7,8:8,9:9,10:10,11:11,12:12}, 

    "Hour":
    {0:0, 1:1, 2:2, 3:3,4:4,5:5,6:6,7:7,8:8,9:9,10:10,11:11,12:12,13:13,14:14,15:15,16:16,17:17,18:18,19:19,20:20,21:21,22:22,23:23,24:24}

}


X

In [92]:
# TODO: build X (features) and y (target), then encode the categorical columns
# Hint: y = data[target]
# Hint: X = pd.get_dummies(data.drop(columns=[target]), drop_first=True)


X = pd.get_dummies(data.drop(columns=["Category"]), drop_first=True)

y = data["Category"]


MemoryError: Unable to allocate 318. GiB for an array with shape (878049, 389257) and data type bool

In [ ]:
# TODO: split into training and test sets
# Hint: 
# 
X_train, X_test, y_train, y_test = train_test_split(
           X, y, test_size=0.25, random_state=1, stratify=y)




## Step 3 - Build the models

Each classifier is only a few lines: create it, then call `.fit(X_train, y_train)`.

In [ ]:
# TODO: train a Decision Tree
# Hint: dtree = DecisionTreeClassifier(max_depth=5, random_state=1)
#       dtree.fit(X_train, y_train)


#DECISON TREE DO THIS ONE#

In [ ]:
# TODO: train a Naive Bayes classifier
# Hint: nb_model = GaussianNB()
#       nb_model.fit(X_train, y_train)




## Step 4 - Evaluate and compare

For **each** model, predict on the test set and report:

- accuracy (`accuracy_score`)
- the confusion matrix (`ConfusionMatrixDisplay.from_predictions`)
- precision, recall, and F1 (`classification_report`)

Then compare: which model is more accurate, and what do the precision and recall for the
class you care about tell you?

In [ ]:
# TODO: evaluate the Decision Tree
# Hint: dt_pred = dtree.predict(X_test)
#       print(accuracy_score(y_test, dt_pred))
#       ConfusionMatrixDisplay.from_predictions(y_test, dt_pred, cmap="Blues"); plt.show()
#       print(classification_report(y_test, dt_pred))
#
#



In [ ]:
# TODO: evaluate the Naive Bayes model (same steps as above)




In [ ]:
# TODO: compare the two models' accuracy against a baseline
# Hint: baseline = y_test.value_counts(normalize=True).max()
# A simple bar chart makes the comparison clear.




## Step 5 - Findings

Write your answers here as you work.

1. **Preprocessing / data reduction** - what did you have to clean, drop, encode, or sample, and why?

   *Your answer:*

2. **Data used** - which features did you select as predictors, and why those?

   *Your answer:*

3. **Findings** - what did the analysis reveal about the data itself (patterns, surprises)?

   *Your answer:*

4. **Decision Tree vs Naive Bayes** - which performed better on your task, and what are the
   strengths and shortcomings of each one here?

   *Your answer:*

5. **Challenges** - What was difficult, and how did you handle it?

   *Your answer:*

## Submission checklist

- This notebook (renamed `Last_First_Assignment7.ipynb`), with all cells run and output showing
- Link to your GitHub repository